In [ ]:
import pypsa
from matplotlib import pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import geopandas as gpd
import pandas as pd
import os
# Line2D
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm


In [ ]:
TITLE_SIZE = 20

In [ ]:
n_hr_reactive = pypsa.Network('/Users/kamrantehranchi/Local_Documents/pypsa-usa/workflow/notebooks/CH2/3_24/elec_s210_cl180a_ch180a_ec_lv1.0_PRM-3h_E_mapped_TEP.nc')
n_lr_reactive = pypsa.Network('/Users/kamrantehranchi/Local_Documents/pypsa-usa/workflow/notebooks/CH2/3_24/elec_s210_cl7a_ch180a_ec_lv1.0_PRM-3h_E_mapped_TEP.nc')
n_lr_proactive = pypsa.Network("/Users/kamrantehranchi/Local_Documents/pypsa-usa/workflow/notebooks/CH2/BAU/elec_s210_cl7a_ch180a_ec_lvopt_PRM-3h_E_mapped_TEP.nc")
n_hr_proactive = pypsa.Network('/Users/kamrantehranchi/Local_Documents/pypsa-usa/workflow/notebooks/CH2/3_24/elec_s210_c180a_ec_lvopt_PRM-3h_E.nc')

In [ ]:
regions_path = os.path.join("/Users/kamrantehranchi/Local_Documents/pypsa-usa/workflow/resources/CH2/tamu/SQ/reactive/texas/Geospatial/regions_onshore_s182_ch180a.geojson")
regions = gpd.read_file(regions_path)

regions_lr = gpd.read_file('/Users/kamrantehranchi/Local_Documents/pypsa-usa/workflow/resources/CH2/tamu/SQ/reactive/texas/Geospatial/regions_onshore_s15_ch7a.geojson')

In [ ]:
def plot_capacity_map(
    n: pypsa.Network,
    bus_values: pd.DataFrame,
    line_values: pd.DataFrame,
    link_values: pd.DataFrame,
    regions: gpd.GeoDataFrame,
    ax: plt.axes,
    bus_scale=1,
    line_scale=1,
    title=None,
    flow=None,
    line_colors="teal",
    link_colors="green",
    line_cmap="viridis",
    line_norm=None,
    line_alpha=0.5,
    bus_split_circles=False,
) -> plt.axes:
    """Generic network plotting function for capacity pie charts at each node."""
    line_width = line_values / line_scale
    link_width = link_values / line_scale

    with plt.rc_context({"patch.linewidth": 0.1}):
        n.plot(
            bus_sizes=bus_values / bus_scale,
            bus_colors=n.carriers.color,
            bus_alpha=0.7,
            line_widths=line_width,
            link_widths=0 if link_width.empty else link_width,
            line_colors=line_colors,
            link_colors=link_colors,
            ax=ax,
            margin=0.2,
            color_geomap=True,
            flow=flow,
            line_cmap=line_cmap,
            line_norm=line_norm,
            line_alpha=line_alpha,
            bus_split_circles=bus_split_circles,
        )

    # onshore regions
    regions.boundary.plot(
        ax=ax,
        facecolor="gainsboro",
        edgecolor="white",
        aspect="equal",
        transform=ccrs.PlateCarree(),
        linewidth=1.2,
    )
    ax.set_extent(regions.total_bounds[[0, 2, 1, 3]])

    ax.set_title(title or "Capacity (MW)", fontsize=TITLE_SIZE, pad=20)

    return ax


def plot_network_capacity(
    n: pypsa.Network,
    ax: plt.axes,
    regions: gpd.GeoDataFrame,
    plot_type: str = "base",
    bus_scale: float = 1e3,
    line_scale: float = 1e6,
    line_colors: str ="teal",
    link_colors: str="green",
    line_alpha: float = 0.5,
) -> plt.axes:
    """Plot network capacity for buses, lines, and links.".
    
    Options for plot_type:
    - base: Base network capacities
    - opt: Optimal network capacities
    - new: New network capacities
    """
    groupers = n.statistics.groupers

    if plot_type == "base":
        bus_groups = n.statistics.installed_capacity(groupby=groupers.get_bus_and_carrier, nice_names=False)
        bus_groups = bus_groups[bus_groups.index.get_level_values('component') != 'Line']
        bus_groups = bus_groups.droplevel(0)
        bus_values= bus_groups[n.investment_periods[0]]

        line_values = n.lines.s_nom
        link_values = n.links[n.links.carrier == "AC"].p_nom.replace(to_replace={pd.NA: 0})
        title = "Base Network Capacities"
    
    elif plot_type == "opt":
        bus_groups = n.statistics.optimal_capacity(groupby=groupers.get_bus_and_carrier, nice_names=False)
        bus_groups = bus_groups[bus_groups.index.get_level_values('component') != 'Line']
        bus_groups = bus_groups.droplevel(0)
        bus_values= bus_groups[n.investment_periods[0]]

        line_values = n.lines.s_nom_opt
        link_values = n.links[n.links.carrier == "AC"].p_nom_opt.replace(to_replace={pd.NA: 0})
        title = "Optimal Network Capacities"
    
    elif plot_type == "new":
        bus_pnom = n.statistics.installed_capacity(groupby=groupers.get_bus_and_carrier, nice_names=False)
        bus_pnom = bus_pnom[bus_pnom.index.get_level_values('component') != 'Line']
        bus_pnom = bus_pnom.droplevel(0)


        bus_pnom_opt = n.statistics.optimal_capacity(groupby=groupers.get_bus_and_carrier, nice_names=False)
        bus_pnom_opt = bus_pnom_opt[bus_pnom_opt.index.get_level_values('component') != 'Line']
        bus_pnom_opt = bus_pnom_opt.droplevel(0)

        bus_values = bus_pnom_opt - bus_pnom
        bus_values = bus_values[(bus_values > 0)]
        bus_values= bus_values[n.investment_periods[0]]

        
        line_values = n.lines.s_nom_opt - n.lines.s_nom
        link_pnom = n.links[n.links.carrier == "AC"].p_nom
        link_pnom_opt = n.links[n.links.carrier == "AC"].p_nom_opt
        link_values = link_pnom_opt - link_pnom
        link_values = link_values.replace(to_replace={pd.NA: 0})
        title = "New Network Capacities"
    
    else:
        raise ValueError(f"Unknown plot_type: {plot_type}")

    # Plot data
    plot_capacity_map(
        n=n,
        bus_values=bus_values,
        line_values=line_values,
        link_values=link_values,
        regions=regions,
        ax=ax,
        line_scale=line_scale,
        bus_scale=bus_scale,
        title=title,
        line_colors=line_colors,
        link_colors=link_colors,
        line_alpha=line_alpha,
    )
    return ax

# Transmission Comparisons

## Reactive (Low-Res vs High-Res)

In [ ]:
def get_diff_capacities(n: pypsa.Network):
    groupers = n.statistics.groupers
    bus_pnom = n.statistics.installed_capacity(groupby=groupers.get_bus_and_carrier, nice_names=False)
    bus_pnom = bus_pnom[bus_pnom.index.get_level_values('component') != 'Line']
    bus_pnom = bus_pnom.droplevel(0)


    bus_pnom_opt = n.statistics.optimal_capacity(groupby=groupers.get_bus_and_carrier, nice_names=False)
    bus_pnom_opt = bus_pnom_opt[bus_pnom_opt.index.get_level_values('component') != 'Line']
    bus_pnom_opt = bus_pnom_opt.droplevel(0)

    bus_values = bus_pnom_opt.fillna(0) - bus_pnom.fillna(0)
    bus_values= bus_values[n.investment_periods[0]]


    line_values = n.lines.s_nom_opt - n.lines.s_nom
    link_pnom = n.links[n.links.carrier == "AC"].p_nom
    link_pnom_opt = n.links[n.links.carrier == "AC"].p_nom_opt
    link_values = link_pnom_opt - link_pnom
    link_values = link_values.replace(to_replace={pd.NA: 0})
    return bus_values, line_values, link_values


In [ ]:
iq_diff_bus, iq_diff_line, iq_diff_link = get_diff_capacities(n_hr_reactive)
sq_diff_bus, sq_diff_line, sq_diff_link = get_diff_capacities(n_lr_reactive)

diff_bus = iq_diff_bus - sq_diff_bus 
diff_line = iq_diff_line - sq_diff_line
diff_link = iq_diff_link - sq_diff_link

title = "Diff of Capacities"
line_colors = "purple"
link_colors = "purple"
line_alpha = 0.9
colormap = "seismic"

line_scale = 2e2
bus_scale = 1e10

fig, axes = plt.subplots(1, 1, figsize=(7,7), subplot_kw={"projection": ccrs.EqualEarth(n_lr_reactive.buses.x.mean())})
# Plot data
plot_capacity_map(
    n=n_hr_reactive,
    bus_values=diff_bus,
    line_values=diff_line,
    link_values=diff_link,
    regions=regions,
    ax=axes,
    line_scale=line_scale,
    bus_scale=bus_scale,
    title=title,
    line_colors=diff_line,
    line_cmap=colormap,
    link_colors=link_colors,
    line_alpha=line_alpha,
    bus_split_circles=True
)

plot_network_capacity(
    n_hr_reactive,
    ax=axes,
    regions=regions,
    plot_type='base',
    bus_scale = 10e20,
    line_scale= line_scale,
    line_colors="dimgrey", 
    link_colors="dimgrey",
    line_alpha=0.1,
) 

# Find the maximum absolute value to ensure a symmetric colormap around zero
vmax = max(abs(diff_line.min()), abs(diff_line.max()))
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

# Create ScalarMappable with the centered norm
sm = plt.cm.ScalarMappable(cmap=colormap, norm=norm)
sm._A = []
cbar = plt.colorbar(sm, ax=axes, orientation='vertical', pad=0.02, aspect=50)
cbar.set_label('Capacity Difference (MW)',
                fontsize=TITLE_SIZE * 0.5)

pypsa.plot.add_legend_lines(axes, sizes=[1e3/line_scale, 3e3/line_scale], labels=["1 GW", "3 GW"])

legend_elements = [
    Line2D([0], [0], color='lightgrey', lw=4, label='Existing Network'),
]
axes.legend(handles=legend_elements, loc='upper left', fontsize=TITLE_SIZE * 0.6)

regions_lr.boundary.plot(
    ax=axes,
    facecolor=None,
    edgecolor="black",
    aspect="equal",
    transform=ccrs.PlateCarree(),
    linewidth=0.7,
)

plt.suptitle("Reactive Transmission Planning Comparison", fontsize=TITLE_SIZE)
plt.title("(High Res - Low Res)", fontsize=TITLE_SIZE * 0.7)
fig.tight_layout()

## Low-Res Reactive vs. High-Res Proactive

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(7 , 7), subplot_kw={"projection": ccrs.EqualEarth(n_lr_reactive.buses.x.mean())})

line_scale = 5e2
bus_scale = 1e6

plot_network_capacity(
    n_hr_proactive,
    ax=axes,
    regions=regions,
    plot_type='base',
    bus_scale = 1e10,
    line_scale= line_scale,
    line_colors="dimgrey", 
    link_colors="dimgrey",
    line_alpha=0.1,
) 

SQ_color = "blue"
ID_color = "red"

plot_network_capacity(
    n_lr_reactive,
    ax=axes,
    regions=regions,
    plot_type='new',
    bus_scale = bus_scale,
    line_scale= line_scale,
    line_colors=SQ_color, 
    link_colors=SQ_color,
    line_alpha=0.9,
) 

plot_network_capacity(
    n_hr_proactive,
    ax=axes,
    regions=regions,
    plot_type='new',
    bus_scale = bus_scale,
    line_scale= line_scale,
    line_colors=ID_color, 
    link_colors=ID_color,
    line_alpha=0.4,
) 

pypsa.plot.add_legend_lines(axes, sizes=[1e3/line_scale, 3e3/line_scale], labels=["1 GW", "3 GW"])
legend_elements = [
    Line2D([0], [0], color='lightgrey', lw=4, label='Existing Network'),
    Line2D([0], [0], color=SQ_color, lw=4, label='Status Quo Reactive'),
    Line2D([0], [0], color=ID_color, lw=4, label='Idealized Proactive'),
]
axes.legend(handles=legend_elements, loc='upper left', fontsize=TITLE_SIZE * 0.6)

regions_lr.boundary.plot(
    ax=axes,
    facecolor=None,
    edgecolor="black",
    aspect="equal",
    transform=ccrs.PlateCarree(),
    linewidth=0.7,
)

plt.title("Comparison of Network Capacities", fontsize=TITLE_SIZE, pad=20)
fig.tight_layout()

In [ ]:

iq_diff_bus, iq_diff_line, iq_diff_link = get_diff_capacities(n_hr_proactive)
sq_diff_bus, sq_diff_line, sq_diff_link = get_diff_capacities(n_lr_reactive)


diff_bus = iq_diff_bus - sq_diff_bus 
diff_line = iq_diff_line - sq_diff_line
diff_link = iq_diff_link - sq_diff_link


title = "Diff of Capacities"
line_colors = "purple"
link_colors = "purple"
line_alpha = 0.9
colormap = "seismic"

line_scale = 2e2
bus_scale = 1e6

fig, axes = plt.subplots(1, 1, figsize=(7,7), subplot_kw={"projection": ccrs.EqualEarth(n_lr_reactive.buses.x.mean())})

# Plot data
plot_capacity_map(
    n=n_hr_proactive,
    bus_values=0,
    line_values=diff_line,
    link_values=diff_link,
    regions=regions,
    ax=axes,
    line_scale=line_scale,
    bus_scale=bus_scale,
    title=title,
    line_colors=diff_line,
    line_cmap=colormap,
    link_colors=link_colors,
    line_alpha=line_alpha,
    bus_split_circles=True
)

plot_network_capacity(
    n_hr_reactive,
    ax=axes,
    regions=regions,
    plot_type='base',
    bus_scale = 1e10,
    line_scale= line_scale,
    line_colors="dimgrey", 
    link_colors="dimgrey",
    line_alpha=0.1,
) 

# Find the maximum absolute value to ensure a symmetric colormap around zero
vmax = max(abs(diff_line.min()), abs(diff_line.max()))
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

# Create ScalarMappable with the centered norm
sm = plt.cm.ScalarMappable(cmap=colormap, norm=norm)
sm._A = []
cbar = plt.colorbar(sm, ax=axes, orientation='vertical', pad=0.02, aspect=50)
cbar.set_label('Capacity Difference (MW)',
                fontsize=TITLE_SIZE * 0.5)

pypsa.plot.add_legend_lines(axes, sizes=[1e3/line_scale, 3e3/line_scale], labels=["1 GW", "3 GW"])

legend_elements = [
    Line2D([0], [0], color='lightgrey', lw=4, label='Existing Network'),
]
axes.legend(handles=legend_elements, loc='upper left', fontsize=TITLE_SIZE * 0.6)

regions_lr.boundary.plot(
    ax=axes,
    facecolor=None,
    edgecolor="black",
    aspect="equal",
    transform=ccrs.PlateCarree(),
    linewidth=0.7,
)

plt.suptitle("Difference in Planning Methods", fontsize=TITLE_SIZE)
plt.title("(Idealized Proactive - Status Quo Reactive)", fontsize=TITLE_SIZE * 0.7)
fig.tight_layout()

# Proactive (Low-Res vs High-Res)

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(7 , 7), subplot_kw={"projection": ccrs.EqualEarth(n_lr_reactive.buses.x.mean())})

line_scale = 5e2
bus_scale = 1e6

plot_network_capacity(
    n_hr_proactive,
    ax=axes,
    regions=regions,
    plot_type='base',
    bus_scale = 1e10,
    line_scale= line_scale,
    line_colors="dimgrey", 
    link_colors="dimgrey",
    line_alpha=0.1,
) 

SQ_color = "blue"
ID_color = "red"

plot_network_capacity(
    n_lr_proactive,
    ax=axes,
    regions=regions,
    plot_type='new',
    bus_scale = bus_scale,
    line_scale= line_scale,
    line_colors=SQ_color, 
    link_colors=SQ_color,
    line_alpha=0.9,
) 

plot_network_capacity(
    n_hr_proactive,
    ax=axes,
    regions=regions,
    plot_type='new',
    bus_scale = bus_scale,
    line_scale= line_scale,
    line_colors=ID_color, 
    link_colors=ID_color,
    line_alpha=0.4,
) 

pypsa.plot.add_legend_lines(axes, sizes=[1e3/line_scale, 3e3/line_scale], labels=["1 GW", "3 GW"])
legend_elements = [
    Line2D([0], [0], color='lightgrey', lw=4, label='Existing Network'),
    Line2D([0], [0], color=SQ_color, lw=4, label='Low Res Proactive'),
    Line2D([0], [0], color=ID_color, lw=4, label='High Res Proactive'),
]
axes.legend(handles=legend_elements, loc='upper left', fontsize=TITLE_SIZE * 0.6)

regions_lr.boundary.plot(
    ax=axes,
    facecolor=None,
    edgecolor="black",
    aspect="equal",
    transform=ccrs.PlateCarree(),
    linewidth=0.7,
)

plt.title("Comparison of Network Capacities", fontsize=TITLE_SIZE, pad=20)
fig.tight_layout()

In [ ]:
iq_diff_bus, iq_diff_line, iq_diff_link = get_diff_capacities(n_hr_proactive)
sq_diff_bus, sq_diff_line, sq_diff_link = get_diff_capacities(n_lr_proactive)


diff_bus = iq_diff_bus - sq_diff_bus 
diff_line = iq_diff_line - sq_diff_line
diff_link = iq_diff_link - sq_diff_link


title = "Diff of Capacities"
line_colors = "purple"
link_colors = "purple"
line_alpha = 0.9
colormap = "seismic"

line_scale = 2e2
bus_scale = 1e6

fig, axes = plt.subplots(1, 1, figsize=(7,7), subplot_kw={"projection": ccrs.EqualEarth(n_lr_reactive.buses.x.mean())})

# Plot data
plot_capacity_map(
    n=n_hr_proactive,
    bus_values=0,
    line_values=diff_line,
    link_values=diff_link,
    regions=regions,
    ax=axes,
    line_scale=line_scale,
    bus_scale=bus_scale,
    title=title,
    line_colors=diff_line,
    line_cmap=colormap,
    link_colors=link_colors,
    line_alpha=line_alpha,
    bus_split_circles=True
)

plot_network_capacity(
    n_hr_reactive,
    ax=axes,
    regions=regions,
    plot_type='base',
    bus_scale = 1e10,
    line_scale= line_scale,
    line_colors="dimgrey", 
    link_colors="dimgrey",
    line_alpha=0.1,
) 

# Find the maximum absolute value to ensure a symmetric colormap around zero
vmax = max(abs(diff_line.min()), abs(diff_line.max()))
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

# Create ScalarMappable with the centered norm
sm = plt.cm.ScalarMappable(cmap=colormap, norm=norm)
sm._A = []
cbar = plt.colorbar(sm, ax=axes, orientation='vertical', pad=0.02, aspect=50)
cbar.set_label('Capacity Difference (MW)',
                fontsize=TITLE_SIZE * 0.5)

pypsa.plot.add_legend_lines(axes, sizes=[1e3/line_scale, 3e3/line_scale], labels=["1 GW", "3 GW"])

legend_elements = [
    Line2D([0], [0], color='lightgrey', lw=4, label='Existing Network'),
]
axes.legend(handles=legend_elements, loc='upper left', fontsize=TITLE_SIZE * 0.6)

regions_lr.boundary.plot(
    ax=axes,
    facecolor=None,
    edgecolor="black",
    aspect="equal",
    transform=ccrs.PlateCarree(),
    linewidth=0.7,
)

plt.suptitle("Proactive Transmission Planning Differences", fontsize=TITLE_SIZE)
plt.title("(High Res - Low Res)", fontsize=TITLE_SIZE * 0.7)
fig.tight_layout()

# Multi-plot

In [ ]:
# Reactive differences
iq_diff_bus, iq_diff_line, iq_diff_link = get_diff_capacities(n_hr_reactive)
sq_diff_bus, sq_diff_line, sq_diff_link = get_diff_capacities(n_lr_reactive)
diff_bus_reactive = iq_diff_bus - sq_diff_bus
diff_line_reactive = iq_diff_line - sq_diff_line
diff_link_reactive = iq_diff_link - sq_diff_link

# Proactive differences
iq_diff_bus, iq_diff_line, iq_diff_link = get_diff_capacities(n_hr_proactive)
sq_diff_bus, sq_diff_line, sq_diff_link = get_diff_capacities(n_lr_proactive)
diff_bus_proactive = iq_diff_bus - sq_diff_bus
diff_line_proactive = iq_diff_line - sq_diff_line
diff_link_proactive = iq_diff_link - sq_diff_link

# Create a list of dictionaries for each subplot
subplot_data = [
    {
        'title': 'Reactive Differences (a - b)',
        'bus_values': diff_bus_reactive,
        'line_values': diff_line_reactive,
        'link_values': diff_link_reactive
    },
    {
        'title': 'Proactive Differences (c - d)',
        'bus_values': diff_bus_proactive,
        'line_values': diff_line_proactive, 
        'link_values': diff_link_proactive
    },
]


# Common parameters
line_colors = "purple"
link_colors = "purple"
line_alpha = 0.9
colormap = "seismic"
line_scale = 2e2
bus_scale = 1e10

# Create figure with subplots
fig, axes = plt.subplots(1, 2, figsize=(10, 5), 
                         subplot_kw={"projection": ccrs.EqualEarth(n_lr_reactive.buses.x.mean())})

# Find the maximum absolute value across all data for consistent color scaling
vmax = max([
    max(abs(data['line_values'].min()), abs(data['line_values'].max())) 
    for data in subplot_data
])
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

# Plot each subplot
for i, (ax, data) in enumerate(zip(axes, subplot_data)):
    # Plot capacity map
    plot_capacity_map(
        n=n_hr_reactive,
        bus_values=data['bus_values'],
        line_values=data['line_values'],
        link_values=data['link_values'],
        regions=regions,
        ax=ax,
        line_scale=line_scale,
        bus_scale=bus_scale,
        title=None,  # We'll set title separately
        line_colors=data['line_values'],
        line_cmap=colormap,
        link_colors=link_colors,
        line_alpha=line_alpha,
        line_norm=norm,  # Use the same norm for all subplots
        bus_split_circles=True
    )
    
    # Plot network base
    plot_network_capacity(
        n_hr_reactive,
        ax=ax,
        regions=regions,
        plot_type='base',
        bus_scale=10e20,
        line_scale=line_scale,
        line_colors="dimgrey",
        link_colors="dimgrey",
        line_alpha=0.1,
    )
    
    # Add region boundaries
    regions_lr.boundary.plot(
        ax=ax,
        facecolor=None,
        edgecolor="black",
        aspect="equal",
        transform=ccrs.PlateCarree(),
        linewidth=0.7,
    )
    
    # Set subplot title
    ax.set_title(data['title'], fontsize=TITLE_SIZE * 0.7)
    
    # Only add legend to the first subplot
    if i == 0:
        pypsa.plot.add_legend_lines(
            ax, 
            sizes=[1e3/line_scale], 
            labels=["1 GW"], legend_kw={"loc": "lower left", "fontsize": TITLE_SIZE * 0.6}
        )
        legend_elements = [
            Line2D([0], [0], color='lightgrey', lw=4, label='Existing Network'),
        ]
        ax.legend(handles=legend_elements, loc='upper left', fontsize=TITLE_SIZE * 0.6)

# Create a ScalarMappable for the colorbar
sm = plt.cm.ScalarMappable(cmap=colormap, norm=norm)
sm._A = []

# Add a colorbar that applies to all subplots
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='vertical')
cbar.set_label('Capacity Difference (MW)', fontsize=TITLE_SIZE * 0.5)

# Add main title
plt.suptitle("Transmission Investment Decision Comparison", fontsize=TITLE_SIZE)

# Adjust layout
fig.tight_layout(rect=[0, 0, 0.9, 0.95])  # Make room for colorbar

plt.show()

# Plot generation capacities map

In [ ]:
groupers = pypsa.statistics.groupers
def get_capacities(n: pypsa.Network):
    groupers = pypsa.statistics.groupers
    bus_capacities = n.statistics.optimal_capacity(groupby=groupers["bus", "carrier"], nice_names=False)
    bus_capacities = bus_capacities[bus_capacities.index.get_level_values('component') != 'Line']
    bus_capacities = bus_capacities.droplevel(0)
    bus_capacities = bus_capacities[n.investment_periods[0]]
    return bus_capacities

cap_id = get_capacities(n_hr_reactive)
cap_sq = get_capacities(n_lr_reactive)
diff_cap = cap_id - cap_sq

iq_diff_bus, iq_diff_line, iq_diff_link = get_diff_capacities(n_hr_reactive)
sq_diff_bus, sq_diff_line, sq_diff_link = get_diff_capacities(n_lr_reactive)

diff_line = iq_diff_line - sq_diff_line
diff_link = iq_diff_link - sq_diff_link

fig , axes = plt.subplots(1, 1, figsize=(7,7), subplot_kw={"projection": ccrs.EqualEarth(n_lr_reactive.buses.x.mean())})
# Plot data
bus_scale = 1e5

plot_capacity_map(
    n_hr_reactive,
    ax=axes,
    regions=regions,
    bus_values=diff_cap,
    link_values=diff_link,
    line_values=diff_line,
    bus_scale = 1e5,
    line_scale= line_scale,
    title="",
    link_colors="green",
    line_norm=None,
    line_alpha=0.5,
    bus_split_circles=True,
    line_colors=diff_line,
    line_cmap=colormap,
)

regions_lr.boundary.plot(
    ax=axes,
    facecolor=None,
    edgecolor="black",
    aspect="equal",
    transform=ccrs.PlateCarree(),
    linewidth=0.7,
)

plt.suptitle("Difference in Optimal Capacities", fontsize=TITLE_SIZE)
plt.title("(Idealized - Status Quo)", fontsize=TITLE_SIZE * 0.7)

pypsa.plot.add_legend_semicircles(axes, sizes=[1e3/bus_scale, -1e3/bus_scale], labels=["1 GW Increase", "1 GW decrease"])

In [ ]:
groupers = pypsa.statistics.groupers
def get_capacities(n: pypsa.Network):
    groupers = pypsa.statistics.groupers
    bus_capacities = n.statistics.optimal_capacity(groupby=groupers["bus", "carrier"], nice_names=False)
    bus_capacities = bus_capacities[bus_capacities.index.get_level_values('component') != 'Line']
    bus_capacities = bus_capacities.droplevel(0)
    bus_capacities = bus_capacities[n.investment_periods[0]]
    return bus_capacities

cap_id = get_capacities(n_hr_proactive)
cap_sq = get_capacities(n_hr_reactive)
diff_cap = cap_id - cap_sq

iq_diff_bus, iq_diff_line, iq_diff_link = get_diff_capacities(n_hr_proactive)
sq_diff_bus, sq_diff_line, sq_diff_link = get_diff_capacities(n_hr_reactive)

diff_line = iq_diff_line - sq_diff_line
diff_link = iq_diff_link - sq_diff_link

fig , axes = plt.subplots(1, 1, figsize=(7,7), subplot_kw={"projection": ccrs.EqualEarth(n_lr_reactive.buses.x.mean())})
# Plot data
bus_scale = 1e5

plot_capacity_map(
    n_hr_reactive,
    ax=axes,
    regions=regions,
    bus_values=diff_cap,
    link_values=diff_link,
    line_values=diff_line,
    bus_scale = 1e5,
    line_scale= line_scale,
    title="",
    link_colors="green",
    line_norm=None,
    line_alpha=0.5,
    bus_split_circles=True,
    line_colors=diff_line,
    line_cmap=colormap,
)

regions_lr.boundary.plot(
    ax=axes,
    facecolor=None,
    edgecolor="black",
    aspect="equal",
    transform=ccrs.PlateCarree(),
    linewidth=0.7,
)

plt.suptitle("Difference in Optimal Capacities", fontsize=TITLE_SIZE)
plt.title("(Proactive - Status Quo)", fontsize=TITLE_SIZE * 0.7)

pypsa.plot.add_legend_semicircles(axes, sizes=[1e3/bus_scale, -1e3/bus_scale], labels=["1 GW Increase", "1 GW decrease"])

# Production Change (Reactive)

In [ ]:

groupers = pypsa.statistics.groupers
def get_supply(n: pypsa.Network):
    groupers = pypsa.statistics.groupers
    bus_supply = n.statistics.supply(groupby=groupers["bus", "carrier"], nice_names=False)
    bus_supply = bus_supply[bus_supply.index.get_level_values('component') != 'Line']
    bus_supply = bus_supply.droplevel(0)
    bus_supply = bus_supply[n.investment_periods[0]]
    return bus_supply

In [ ]:
supply_id = get_supply(n_hr_reactive)
supply_sq = get_supply(n_lr_reactive)
diff_supply = supply_id - supply_sq

iq_diff_bus, iq_diff_line, iq_diff_link = get_diff_capacities(n_hr_reactive)
sq_diff_bus, sq_diff_line, sq_diff_link = get_diff_capacities(n_lr_reactive)
 
diff_line = iq_diff_line - sq_diff_line
diff_link = iq_diff_link - sq_diff_link


fig , axes = plt.subplots(1, 1, figsize=(7,7), subplot_kw={"projection": ccrs.EqualEarth(n_lr_reactive.buses.x.mean())})
# Plot data
bus_scale = 1e8



plot_network_capacity(
    n_hr_reactive,
    ax=axes,
    regions=regions,
    plot_type='base',
    bus_scale = 1e20,
    line_scale= line_scale,
    line_colors="dimgrey", 
    link_colors="dimgrey",
    line_alpha=0.4,
) 


plot_capacity_map(
    n_hr_reactive,
    ax=axes,
    regions=regions,
    bus_values=diff_supply,
    link_values=diff_link,
    line_values=diff_line,
    bus_scale = bus_scale,
    line_scale= line_scale,
    title="",
    link_colors="green",
    line_norm=None,
    line_alpha=1,
    bus_split_circles=True,
    line_colors=diff_line,
    line_cmap=colormap,
)

regions_lr.boundary.plot(
    ax=axes,
    facecolor=None,
    edgecolor="black",
    aspect="equal",
    transform=ccrs.PlateCarree(),
    linewidth=0.7,
)

plt.suptitle("Difference in Supply - Reactive Planning", fontsize=TITLE_SIZE)
plt.title("(High Res - Low Res)", fontsize=TITLE_SIZE * 0.7)

pypsa.plot.add_legend_semicircles(axes, sizes=[1e6/bus_scale, -1e6/bus_scale], labels=["1 TWh Increase", "1 TWh decrease"], legend_kw={"loc": "upper left"})

In [ ]:
supply_id = get_supply(n_hr_proactive)
supply_sq = get_supply(n_lr_proactive)
diff_supply = supply_id - supply_sq

iq_diff_bus, iq_diff_line, iq_diff_link = get_diff_capacities(n_hr_proactive)
sq_diff_bus, sq_diff_line, sq_diff_link = get_diff_capacities(n_lr_proactive)
 
diff_line = iq_diff_line - sq_diff_line
diff_link = iq_diff_link - sq_diff_link


fig , axes = plt.subplots(1, 1, figsize=(7,7), subplot_kw={"projection": ccrs.EqualEarth(n_lr_proactive.buses.x.mean())})
# Plot data
bus_scale = 1e8

plot_network_capacity(
    n_hr_reactive,
    ax=axes,
    regions=regions,
    plot_type='base',
    bus_scale = 1e20,
    line_scale= line_scale,
    line_colors="dimgrey", 
    link_colors="dimgrey",
    line_alpha=0.4,
) 


plot_capacity_map(
    n_hr_reactive,
    ax=axes,
    regions=regions,
    bus_values=diff_supply,
    link_values=diff_link,
    line_values=diff_line,
    bus_scale = bus_scale,
    line_scale= line_scale,
    title="",
    link_colors="green",
    line_norm=None,
    line_alpha=1,
    bus_split_circles=True,
    line_colors=diff_line,
    line_cmap=colormap,
)

regions_lr.boundary.plot(
    ax=axes,
    facecolor=None,
    edgecolor="black",
    aspect="equal",
    transform=ccrs.PlateCarree(),
    linewidth=0.7,
)

plt.suptitle("Proactive Transmission Planning", fontsize=TITLE_SIZE)
plt.title("(High Resolution - Low Resolution)", fontsize=TITLE_SIZE * 0.7)

In [ ]:
supply_id = get_supply(n_hr_proactive)
supply_sq = get_supply(n_hr_reactive)
diff_supply = supply_id - supply_sq

iq_diff_bus, iq_diff_line, iq_diff_link = get_diff_capacities(n_hr_proactive)
sq_diff_bus, sq_diff_line, sq_diff_link = get_diff_capacities(n_hr_reactive)
 
diff_line = iq_diff_line - sq_diff_line
diff_link = iq_diff_link - sq_diff_link


fig , axes = plt.subplots(1, 1, figsize=(7,7), subplot_kw={"projection": ccrs.EqualEarth(n_hr_reactive.buses.x.mean())})
# Plot data
bus_scale = 1e8

plot_network_capacity(
    n_hr_reactive,
    ax=axes,
    regions=regions,
    plot_type='base',
    bus_scale = 1e20,
    line_scale= line_scale,
    line_colors="dimgrey", 
    link_colors="dimgrey",
    line_alpha=0.4,
) 

plot_capacity_map(
    n_hr_reactive,
    ax=axes,
    regions=regions,
    bus_values=diff_supply,
    link_values=diff_link,
    line_values=diff_line,
    bus_scale = bus_scale,
    line_scale= line_scale,
    title="",
    link_colors="green",
    line_norm=None,
    line_alpha=1,
    bus_split_circles=True,
    line_colors=diff_line,
    line_cmap=colormap,
)

regions_lr.boundary.plot(
    ax=axes,
    facecolor=None,
    edgecolor="black",
    aspect="equal",
    transform=ccrs.PlateCarree(),
    linewidth=0.7,
)

plt.suptitle("Proactive vs Reactive Planning", fontsize=TITLE_SIZE)
plt.title("(High Resolution Proactive - Low Resolution Reactive)", fontsize=TITLE_SIZE * 0.7)

In [ ]:
# plot total supply
groupers = pypsa.statistics.groupers
def get_supply(n: pypsa.Network):
    groupers = pypsa.statistics.groupers
    bus_supply = n.statistics.supply(groupby=groupers["bus", "carrier"], nice_names=False)
    bus_supply = bus_supply[bus_supply.index.get_level_values('component') != 'Line']
    bus_supply = bus_supply.droplevel(0)
    bus_supply = bus_supply[n.investment_periods[0]]
    return bus_supply

cap_id = get_supply(n_hr_reactive)
cap_sq = get_supply(n_lr_reactive)

fig , axes = plt.subplots(1, 1, figsize=(7,7), subplot_kw={"projection": ccrs.EqualEarth(n_lr_reactive.buses.x.mean())})

bus_scale = 1e6
bus_scale = 1e8
line_scale = 1e3


plot_network_capacity(
    n_hr_reactive,
    ax=axes,
    regions=regions,
    plot_type='base',
    bus_scale = 1e20,
    line_scale= line_scale,
    line_colors="dimgrey", 
    link_colors="dimgrey",
    line_alpha=1,
) 

plot_capacity_map(
    n_hr_reactive,
    ax=axes,
    regions=regions,
    bus_values=cap_id,
    link_values=diff_link,
    line_values=n_hr_reactive.lines.s_nom_opt,
    bus_scale = bus_scale,
    line_scale= line_scale,
    title="Total Supply",
    link_colors="green",
    line_norm=None,
    line_alpha=1,
    bus_split_circles=False,
    line_colors="green",
)

regions_lr.boundary.plot(
    ax=axes,
    facecolor=None,
    edgecolor="black",
    aspect="equal",
    transform=ccrs.PlateCarree(),
    linewidth=0.7,
)

pypsa.plot.add_legend_semicircles(axes, sizes=[1e6/bus_scale, -1e6/bus_scale], labels=["1 TWh Increase", "1 TWh decrease"], legend_kw={"loc": "upper left"})

# Inter-regional vs Local Transmission Development

In [ ]:
# Function to mark lines as interregional or intraregional
def mark_interregional_lines(network):
    network.lines['bus0_zone'] = network.lines['bus0'].map(lambda x: network.buses.loc[x, 'reeds_zone'])
    network.lines['bus1_zone'] = network.lines['bus1'].map(lambda x: network.buses.loc[x, 'reeds_zone'])
    network.lines['interregional'] = network.lines['bus0_zone'] != network.lines['bus1_zone']
    return network
# Mark interregional lines for each network
n_hr_reactive = mark_interregional_lines(n_hr_reactive)
n_lr_reactive = mark_interregional_lines(n_lr_reactive)
n_lr_proactive = mark_interregional_lines(n_lr_proactive)
n_hr_proactive = mark_interregional_lines(n_hr_proactive)


# Calculate the proportion of interregional vs. intraregional transmission capacity
def calculate_transmission_proportions(network):
    # Calculate new capacity for each line (only positive values)
    new_capacity = (network.lines.s_nom_opt - network.lines.s_nom).clip(lower=0)  / 1e3  # Convert to GW
    new_capacity_length = network.lines.length * new_capacity
    
    # Calculate total new capacity
    total_new_capacity = new_capacity_length.sum()
    
    # Calculate interregional and intraregional capacity
    interregional_capacity = new_capacity_length[network.lines.interregional].sum()
    intraregional_capacity = new_capacity_length[~network.lines.interregional].sum()
    
    return {
        'total_capacity': total_new_capacity,
        'interregional_capacity': interregional_capacity,
        'intraregional_capacity': intraregional_capacity
    }

# Calculate proportions for each network
lr_reactive_prop = calculate_transmission_proportions(n_lr_reactive)
hr_reactive_prop = calculate_transmission_proportions(n_hr_reactive)
lr_proactive_proportions = calculate_transmission_proportions(n_lr_proactive)
proactive_proportions = calculate_transmission_proportions(n_hr_proactive)

# Create the plot
fig, ax = plt.subplots(figsize=(7, 5))

# Set up the data
networks = ['LowRes Reactive', 'HighRes Reactive', 'LowRes Proactive' , 'HighRes Proactive']
interregional_caps = [lr_reactive_prop['interregional_capacity'], hr_reactive_prop['interregional_capacity'],lr_proactive_proportions['interregional_capacity'], proactive_proportions['interregional_capacity']]
intraregional_caps = [lr_reactive_prop['intraregional_capacity'], hr_reactive_prop['intraregional_capacity'],lr_proactive_proportions['intraregional_capacity'],proactive_proportions['intraregional_capacity']]
total_caps = [lr_reactive_prop['total_capacity'], hr_reactive_prop['total_capacity'],lr_proactive_proportions['total_capacity'], proactive_proportions['total_capacity']]

# Create the stacked bar plot
x = np.arange(len(networks))
width = 0.6

ax.bar(x, interregional_caps, width, label='Interregional', color='darkblue')
ax.bar(x, intraregional_caps, width, bottom=interregional_caps, label='Local', color='lightblue')

# Add labels and title
ax.set_xlabel('Network', fontsize=TITLE_SIZE * 0.7)
ax.set_ylabel('New Capacity (GW-mi)', fontsize=TITLE_SIZE * 0.7)
ax.set_title('Interregional vs. Local Transmission Capacity', fontsize=TITLE_SIZE)
ax.set_xticks(x)
ax.set_xticklabels(networks)
ax.legend(loc='upper right')

# Add text labels with absolute values
for i in range(len(networks)):
    # Text color based on background darkness
    interregional_color = intraregional_color = 'white'

    # Add absolute value labels
    ax.text(i, interregional_caps[i]/2, f'{interregional_caps[i]:.0f} GW-mi', 
             ha='center', va='center', fontsize=TITLE_SIZE * 0.5, color=interregional_color)
    ax.text(i, interregional_caps[i] + intraregional_caps[i]/2, f'{intraregional_caps[i]:.0f} GW-mi', 
             ha='center', va='center', fontsize=TITLE_SIZE * 0.5, color=intraregional_color)

# Set y-axis limit to make room for the total capacity text
y_max = max([interregional_caps[i] + intraregional_caps[i] for i in range(len(networks))]) * 1.2
ax.set_ylim(0, y_max)

plt.tight_layout()

# LMP Comparison

In [ ]:
# Extract marginal price data from both networks
hr_reactive_prices = n_hr_reactive.buses_t.marginal_price
lr_reactive_prices = n_lr_reactive.buses_t.marginal_price

# Proactive
lr_proactive_prices = n_lr_proactive.buses_t.marginal_price
hr_proactive_prices = n_hr_proactive.buses_t.marginal_price

# Calculate mean prices across all buses for each timestamp
id_mean_price = hr_reactive_prices.mean(axis=1).fillna(0)
sq_mean_price = lr_reactive_prices.mean(axis=1).fillna(0)

# Sort prices in descending order
id_sorted = id_mean_price.sort_values(ascending=False)
sq_sorted = sq_mean_price.sort_values(ascending=False)

# Create percentage index (0-100%)
percentiles = np.linspace(0, 100, len(id_sorted))

# Calculate statistics
id_avg = id_mean_price.mean()
sq_avg = sq_mean_price.mean()

# Sort each bus's prices to align with the overall sorted order
# For ID network
id_prices_sorted = []
id_percentiles = {}
for col in hr_reactive_prices.columns:
    id_prices_sorted.append(hr_reactive_prices[col].loc[id_sorted.index].values)
id_prices_sorted = np.array(id_prices_sorted)
id_25th = np.percentile(id_prices_sorted, 25, axis=0)
id_75th = np.percentile(id_prices_sorted, 75, axis=0)

# For SQ network
sq_prices_sorted = []
sq_percentiles = {}
for col in lr_reactive_prices.columns:
    sq_prices_sorted.append(lr_reactive_prices[col].loc[sq_sorted.index].values)
sq_prices_sorted = np.array(sq_prices_sorted)
sq_25th = np.percentile(sq_prices_sorted, 25, axis=0)
sq_75th = np.percentile(sq_prices_sorted, 75, axis=0)

# Create the figure and plot
fig, ax = plt.subplots(figsize=(10, 6))

# Set reasonable y-axis limits to focus on the relevant price range
y_upper = max(np.percentile(id_sorted, 99), np.percentile(sq_sorted, 98))

# Plot the duration curves
ax.plot(percentiles, id_sorted, color=ID_color, linewidth=2, label=f'Idealized (avg: ${id_avg:.2f}/MWh)')
ax.plot(percentiles, sq_sorted, color=SQ_color, linewidth=2, label=f'Status Quo (avg: ${sq_avg:.2f}/MWh)')

# Add labels and title
ax.set_xlabel('Duration (%)', fontsize=TITLE_SIZE*0.7)
ax.set_ylabel('Marginal Price ($/MWh)', fontsize=TITLE_SIZE*0.7)
ax.set_title('Price Duration Curve Comparison', fontsize=TITLE_SIZE)

# Set y-axis limits
ax.set_ylim(0, y_upper)

# Add a grid for better readability
ax.grid(True, linestyle='--', alpha=0.7)

# Add a legend
ax.legend(fontsize=TITLE_SIZE*0.6)

# Adjust layout
fig.tight_layout()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define network combinations to plot
networks = {
    'HR Reactive': {'data': n_hr_reactive, 'color': ID_color, 'linestyle': '-'},
    'LR Reactive': {'data': n_lr_reactive, 'color': SQ_color, 'linestyle': '-'},
    'HR Proactive': {'data': n_hr_proactive, 'color': ID_color, 'linestyle': '--'},
    'LR Proactive': {'data': n_lr_proactive, 'color': SQ_color, 'linestyle': '--'}
}

# Create figure
fig, ax = plt.subplots(figsize=(8, 5))

# Process each network
price_data = {}
for name, net_info in networks.items():
    # Extract and process data
    prices = net_info['data'].buses_t.marginal_price
    mean_prices = prices.mean(axis=1).fillna(0)
    sorted_prices = mean_prices.sort_values(ascending=False)
    avg_price = mean_prices.mean()
    
    # Store for later use
    price_data[name] = {
        'sorted': sorted_prices,
        'avg': avg_price
    }
    
    # Plot duration curve
    percentiles = np.linspace(0, 100, len(sorted_prices))
    ax.plot(
        percentiles, 
        sorted_prices, 
        color=net_info['color'], 
        linestyle=net_info['linestyle'],
        linewidth=2, 
        label=f'{name} (avg: ${avg_price:.2f}/MWh)'
    )

# Set y-axis limit using 99th percentile of all data
y_values = [data['sorted'].values for data in price_data.values()]
y_upper = max([np.percentile(y, 99) for y in y_values])

# Configure plot
ax.set_xlabel('Duration (%)', fontsize=TITLE_SIZE*0.7)
ax.set_ylabel('Average LMP ($/MWh)', fontsize=TITLE_SIZE*0.7)
ax.set_title('Price Duration Curve Comparison', fontsize=TITLE_SIZE)
ax.set_ylim(0, y_upper)
ax.grid(True, linestyle='--', alpha=0.7)
ax.legend(fontsize=TITLE_SIZE*0.6)

fig.tight_layout()

# Base Plot

In [ ]:
# groupers = pypsa.statistics.groupers
# bus_capacities = network_proactive.statistics.installed_capacity(groupby=groupers["bus", "carrier"], nice_names=False)
# bus_capacities = bus_capacities[bus_capacities.index.get_level_values('component') != 'Line']
# bus_capacities = bus_capacities.droplevel(0)
# bus_capacities = bus_capacities[network_proactive.investment_periods[0]]

# fig , axes = plt.subplots(1, 1, figsize=(7,7), subplot_kw={"projection": ccrs.EqualEarth(network_SQ.buses.x.mean())})
# # Plot data
# bus_scale = 1e5

# plot_capacity_map(
#     network_proactive,
#     ax=axes,
#     regions=regions,
#     bus_values=bus_capacities,
#     link_values=diff_link,
#     line_values=diff_line,
#     bus_scale = 1e5,
#     line_scale= line_scale,
#     title="Difference in Optimal Capacities",
#     link_colors="green",
#     line_norm=None,
#     line_alpha=0.5,
#     bus_split_circles=False,
#     line_colors=diff_line,
#     line_cmap=colormap,
# )

# pypsa.plot.add_legend_semicircles(axes, sizes=[1e3/bus_scale, -1e3/bus_scale], labels=["1 GW Increase", "1 GW decrease"])


fig , axes = plt.subplots(1, 1, figsize=(7,7), subplot_kw={"projection": ccrs.EqualEarth(n_lr_reactive.buses.x.mean())})
# Plot data
bus_scale = 1e10
line_scale = 1e3

plot_network_capacity(
    n_hr_proactive,
    ax=axes,
    regions=regions,
    plot_type='new',
    bus_scale = bus_scale,
    line_scale= line_scale,
    line_colors="teal", 
    link_colors="dimgrey",
    line_alpha=0.8,
)